# LFM Semantic Segmentation GFFT Workflow
This notebook trains a GFFT/Fourier-VQ MultiMAE semantic segmentation model for crater-vs-background mask prediction. It loads a split semantic segmentation dataset, builds the GFFT datamodule and TerraTorch segmentation task, runs fine-tuning, writes checkpoints, and creates validation prediction plots.

## Purpose of this notebook
Use this notebook as the active interactive GFFT semantic-segmentation training workflow. The values in the **User Configuration** section mirror the most commonly changed command-line options; lower-level options stay on the centralized experiment-config defaults unless they are explicitly promoted into that section.

**Note**: the default configuration assumes single-band NAC data and a GFFT YAML with NAC normalization stats and backbone checkpoint metadata. For WAC data, change `DATASET_MODALITY` to `"wac"`, set `BAND_FILTER` to the WAC band indices you want to use, and point `GFFT_CONFIG_PATH` at the matching WAC GFFT YAML.


## Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

import lfm

from functools import partialmethod
from glob import glob
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import torch
from lightning.pytorch import seed_everything
from tqdm import tqdm

tqdm.__init__ = partialmethod(tqdm.__init__, disable=False)

In [ ]:
repo_root = Path.cwd().parent
NOTEBOOK_DIR = repo_root / "notebooks"

if not (repo_root / "lfm").exists():
  raise FileNotFoundError(
      "Cannot find lfm/ directory. Run this notebook from "
      "lfm/notebooks/full_model or update repo_root."
  )

sys.path.insert(0, str(repo_root))

from lfm.all_models.all_tasks.utils import (
    create_timestamped_output_dir,
    save_prediction_cache,
)
from lfm.all_models.sem_seg import build_gfft_notebook_configs
from lfm.all_models.sem_seg.plotting import plot_prediction_cache_comparison
from lfm.full_model.sem_seg import semantic_gfft_components

print("Successfully imported LFM modules")

## User Configuration

These are the values a notebook user is expected to edit for a normal GFFT semantic-segmentation run.

### Paths
`BASE_OUTPUT_DIR`: parent directory for timestamped notebook outputs. Checkpoints, config files, prediction caches, and plots are written under a new timestamped subdirectory.

`GFFT_CONFIG_PATH`: TerraTorch-style GFFT YAML. The notebook reads the backbone checkpoint path and pretraining normalization stats from this file.

`GFFT_BACKBONE_CHECKPOINT`: optional explicit GFFT backbone checkpoint override. Leave as `None` to use the checkpoint path from `GFFT_CONFIG_PATH`.

`LIGHTNING_CHECKPOINT`: optional GFFT Lightning checkpoint to resume from. Leave as `None` for a fresh fine-tune.

### Data Selection
To switch datasets, replace `DATA_DICT` with the matching dictionary from the README, under **"Dataset Specifications"**. If you'd like to use your own dataset (not found in the Dataset Specifications section of the README), reference `docs/dataset_contribution.md`.

`DATA_DICT`: dataset-level dictionary that controls the data-specific parts of training. It defines the dataset modality, dataset directory location, chip and label glob patterns, and selected chip bands. Normalization modality and semantic label source are inferred from `dataset_modality` and the label files. 

`DATA_DICT["dataset_name"]`: dataset name for readability. Has no effect on model/dataset functionality. 

`DATA_DICT["data_dir"]`: path to dataset root. This root will have train/val/test folders nested under it.

`DATA_DICT["dataset_modality"]`: modality of the dataset ("wac" or "nac" for currently supported datasets).

`DATA_DICT["image_glob"]/["label_glob"]`: filename pattern to use when finding images/labels in dataset. 

`DATA_DICT["band_filters"]`: modality-local band selection. For WAC, `"vis": [0, 1, 2, 3, 4]` and `"uv": [0, 1]` selects all 7 stored WAC channels. For NAC PHO or DTM, use `[0]` because each modality is stored as a single band. If you would like to remove either "vis" or "uv" modality for WAC, you can delete that entry (so if I wanted only "wac", I would delete the "uv" line entirely).

`MAX_TRAIN_SAMPLES`, `MAX_VAL_SAMPLES`, `MAX_TEST_SAMPLES`: optional split caps for quick experiments. Set any of these to `None` to use the full split. If less samples are found than the max amount, then the actual sample number will be used. 

### Training And Visualization
`BATCH_SIZE`: GFFT training batch size.

`MAX_EPOCHS`: number of fine-tuning epochs.

`GFFT_SHAPE_LOSS_WEIGHT`, `GFFT_SHAPE_LOSS_PAD_FRAC`: semantic shape-loss settings. Set `GFFT_SHAPE_LOSS_WEIGHT=0.0` to disable shape-loss contribution.

`PROGRESS_LOG_EVERY_N_BATCHES`: extra print-based training logger. Set to `0` in notebooks for progress-bar-only output. 

### Defaults Kept In Code
The notebook leaves these centralized defaults unchanged unless you intentionally add overrides to the config cell. File suffixes are inferred automatically from common names such as `_input_nac_chip`, `_input_wac_chip`, `_label`, `_mask`, and `_img`; add explicit `image_suffix` or `label_suffix` only for unusual datasets. Normalization source defaults to `"pretrain"`. A seed of 42 is used for random number generators, to ensure reproducibility. 

In [ ]:
BASE_OUTPUT_DIR = NOTEBOOK_DIR / "outputs" / "semantic_gfft_finetuning"
GFFT_CONFIG_PATH = repo_root / "graha-lunar-fm" / "terratorch_integration" / "configs" / "nac_craters" / "crater_detection_nac_only_fvqmultimae.yaml"
GFFT_BACKBONE_CHECKPOINT = None
LIGHTNING_CHECKPOINT = None

# Data dictionary; see README under "Dataset Specifications" for examples
DATA_DICT = {
    "dataset_name": "nac_imp_old",  # Human-readable dataset name (has no functional effect)
    "data_dir": "/explore/nobackup/projects/lfm/model_inputs/256_256_inputs/imp/old_imp/imp_sem_seg_clean_targets_only",   # Data directory on /explore
    "dataset_modality": "nac",  # Dataset modality
    "image_glob": "*.tif",  # Image (chip) filename pattern the dataset will look for
    "label_glob": "*_label.npz",   # Label (mask) filename pattern the dataset will look for
    "band_filters": {  # Band filters for each modality
        "pho": [0],  # PHO channels to use (0-indexed)
    },
}

# Upper limit for sample counts in train/val/test datasets; if less samples are found, that amount will be used instead
MAX_TRAIN_SAMPLES = 500
MAX_VAL_SAMPLES = 500
MAX_TEST_SAMPLES = 500

BATCH_SIZE = 8  # Number of inputs fed into the model per iteration; often a multiple of 8 for parallelization purposes
MAX_EPOCHS = 1  # Maximum number of epochs, default to 1 for demo purposes. Finetuning typically uses 50-100 epochs, but sometimes less can work.

# General training parameters
GFFT_BACKBONE_LR = 5.0e-5  # Learning rate for Graha FM backbone (lower than head typically)
GFFT_HEAD_LR = 2.0e-4  # Learning rate for task decoder (higher than backbone typically)
GFFT_LAYER_DECAY = 0.75  # Decays learning rate further toward the backbone; allows for more gentle tuning of Graha backbone weights
GFFT_WEIGHT_DECAY = 0.05  # Penalizes large model weights during training, helping model generalize to non-training data
GFFT_WARMUP_STEPS = 500  # Number of optimizer steps before LR scheduler warmup ends

# Semantic segmentation-specific loss parameters
GFFT_SHAPE_LOSS_WEIGHT = 0.05  # Weighting of "shape loss"; higher numbers penalize non-circular predictions more
GFFT_SHAPE_LOSS_PAD_FRAC = 0.3  # Fractional padding around target boxes used by shape loss

PROGRESS_LOG_EVERY_N_BATCHES = 0  # 0 keeps notebook output to Lightning's progress bar only

The configuration cell above mirrors the active GFFT semantic-segmentation training settings. Values not listed there use the centralized defaults documented in the previous markdown cell.


In [ ]:
OUTPUT_DIR = create_timestamped_output_dir(BASE_OUTPUT_DIR)

notebook_configs = build_gfft_notebook_configs(
    output_dir=OUTPUT_DIR,
    base_output_dir=OUTPUT_DIR,
    graha_base_output_dir=OUTPUT_DIR,
    graha_lightning_checkpoint=LIGHTNING_CHECKPOINT,
    gfft_config_path=GFFT_CONFIG_PATH,
    gfft_backbone_checkpoint=GFFT_BACKBONE_CHECKPOINT,
    data_dict=DATA_DICT,
    max_epochs=MAX_EPOCHS,
    graha_batch_size=BATCH_SIZE,
    graha_num_workers=NUM_WORKERS,
    max_train_samples=MAX_TRAIN_SAMPLES,
    max_val_samples=MAX_VAL_SAMPLES,
    max_test_samples=MAX_TEST_SAMPLES,
    graha_shape_loss_weight=GFFT_SHAPE_LOSS_WEIGHT,
    graha_shape_loss_pad_frac=GFFT_SHAPE_LOSS_PAD_FRAC,
    progress_log_every_n_batches=PROGRESS_LOG_EVERY_N_BATCHES,
)

config = notebook_configs.experiment_config
gfft_config = notebook_configs.gfft_config
deps = notebook_configs.dependencies

seed_everything(config.seed)

print("Config created successfully")
print(f"Data dir: {config.data_root}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Normalization modality: {config.normalization_modality}")
print(f"GFFT config YAML: {gfft_config.gfft_config_path}")
print(f"GFFT backbone weights: {gfft_config.backbone_weights}")
print(f"GFFT modality mode: {config.graha_input_modality_mode}")
print(f"Normalization modality: {config.normalization_modality}")

Set PyTorch device to CUDA (GPU-accelerated) if possible

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Output Directory
The timestamped output directory was created while building the config. It contains checkpoints, prediction caches, and plots for this run.


In [ ]:
print(f"Notebook output directory: {OUTPUT_DIR}")

## Create Datamodule
1. Select the semantic datamodule class based on `SEMANTIC_LABEL_SOURCE`.
2. Load GFFT normalization statistics.
3. Create the datamodule and inspect one training batch.


In [ ]:
datamodule_cls = deps[
    "LunarSemanticFromInstanceDatamodule"
    if config.semantic_label_source == "instance"
    else "LunarSemanticMaskSegmentationDatamodule"
]

print("STEP 1: Loading normalization stats...")
print("=" * 60)

means, stds = semantic_gfft_components.get_normalization_stats(
    gfft_config,
    datamodule_cls,
)

print("Done.")

In [ ]:
print("STEP 2: Creating datamodule and inspecting one training batch...")
print("=" * 60)

gfft_datamodule = semantic_gfft_components.create_datamodule(
    gfft_config,
    datamodule_cls,
    means,
    stds,
)
gfft_sample_batch = semantic_gfft_components.inspect_batch(gfft_datamodule)

print("Done.")

## Create TerraTorch Task


In [ ]:
task_cls = semantic_gfft_components.make_downstream_shape_segmentation_task_class(
    deps["LunarShapeSegmentationTask"]
)

gfft_task = semantic_gfft_components.create_task(
    gfft_config,
    task_cls,
    gfft_sample_batch,
)
semantic_gfft_components.inspect_backbone(gfft_task)

## Run Training

In [ ]:
trainer = semantic_gfft_components.create_trainer(
    gfft_config,
    OUTPUT_DIR,
    deps["ValidationPlotCallback"],
    plot_output_dir=OUTPUT_DIR,
    plots_subdir=Path("plots") / "single_model" / "gfft_model",
    checkpoint_subdir=Path("checkpoints") / "gfft_model",
)

In [ ]:
print("" + "=" * 60)
print("Starting training.")
print("=" * 60)

ckpt_path = (
    str(gfft_config.lightning_checkpoint)
    if gfft_config.lightning_checkpoint is not None
    else None
)
trainer.fit(
    gfft_task,
    datamodule=gfft_datamodule,
    ckpt_path=ckpt_path,
)

print("Finished training.")

## Create And Display Validation Visualizations
Training writes validation plots during `trainer.fit()`. This section also saves a small prediction cache on the configured split and renders a cache-based prediction plot in the notebook.


In [ ]:
prediction_cache = save_prediction_cache(
    task=gfft_task,
    datamodule=gfft_datamodule,
    output_dir=OUTPUT_DIR,
    model_name="gfft",
    split=config.prediction_split,
    n_samples=config.prediction_n_samples,
)

prediction_plot = plot_prediction_cache_comparison(
    {"gfft": prediction_cache},
    OUTPUT_DIR / "plots" / "single_model" / "gfft_model",
    n_samples=config.prediction_n_samples,
    filename=f"{config.prediction_split}_semantic_predictions.png",
)

print(f"Saved prediction plot: {prediction_plot}")

In [ ]:
img = mpimg.imread(prediction_plot)
plt.figure(figsize=(16, 14))
plt.imshow(img)
plt.axis("off")
plt.show()

In [ ]:
del gfft_task, gfft_datamodule, trainer
if torch.cuda.is_available():
    torch.cuda.empty_cache()